In [ ]:
import os

import pandas as pd
from sqlalchemy import text
from sqlalchemy.ext.asyncio import (
    AsyncConnection,
    AsyncSession,
    AsyncSessionTransaction,
    create_async_engine,
)

In [ ]:
async def fetch_all_delaney_processed(connection: AsyncConnection):
    async with AsyncSession(connection).begin() as session_tx:
        session_tx: AsyncSessionTransaction
        session = session_tx.session
        result_cursor = await session.execute(text("SELECT * FROM delaney_processed"))
        result = result_cursor.fetchall()

    return pd.DataFrame(result)


async def fetch_stats_delaney_processed(connection: AsyncConnection):
    async with AsyncSession(connection).begin() as session_tx:
        session_tx: AsyncSessionTransaction
        session = session_tx.session
        sql = """
SELECT 
    'esol_predicted_log_solubility' AS column_name,
    AVG(esol_predicted_log_solubility) AS mean,
    STDDEV(esol_predicted_log_solubility) AS std,
    MIN(esol_predicted_log_solubility) AS min,
    MAX(esol_predicted_log_solubility) AS max,
    percentile_cont(0.25) WITHIN GROUP (ORDER BY esol_predicted_log_solubility) AS p25,
    percentile_cont(0.50) WITHIN GROUP (ORDER BY esol_predicted_log_solubility) AS p50,
    percentile_cont(0.75) WITHIN GROUP (ORDER BY esol_predicted_log_solubility) AS p75
FROM delaney_processed

UNION ALL

SELECT 
    'minimum_degree',
    AVG(minimum_degree), STDDEV(minimum_degree), MIN(minimum_degree), MAX(minimum_degree),
    percentile_cont(0.25) WITHIN GROUP (ORDER BY minimum_degree),
    percentile_cont(0.50) WITHIN GROUP (ORDER BY minimum_degree),
    percentile_cont(0.75) WITHIN GROUP (ORDER BY minimum_degree)
FROM delaney_processed

UNION ALL

SELECT 
    'molecular_weight',
    AVG(molecular_weight), STDDEV(molecular_weight), MIN(molecular_weight), MAX(molecular_weight),
    percentile_cont(0.25) WITHIN GROUP (ORDER BY molecular_weight),
    percentile_cont(0.50) WITHIN GROUP (ORDER BY molecular_weight),
    percentile_cont(0.75) WITHIN GROUP (ORDER BY molecular_weight)
FROM delaney_processed

UNION ALL

SELECT 
    'hbond_donors',
    AVG(hbond_donors), STDDEV(hbond_donors), MIN(hbond_donors), MAX(hbond_donors),
    percentile_cont(0.25) WITHIN GROUP (ORDER BY hbond_donors),
    percentile_cont(0.50) WITHIN GROUP (ORDER BY hbond_donors),
    percentile_cont(0.75) WITHIN GROUP (ORDER BY hbond_donors)
FROM delaney_processed

UNION ALL

SELECT 
    'rings',
    AVG(rings), STDDEV(rings), MIN(rings), MAX(rings),
    percentile_cont(0.25) WITHIN GROUP (ORDER BY rings),
    percentile_cont(0.50) WITHIN GROUP (ORDER BY rings),
    percentile_cont(0.75) WITHIN GROUP (ORDER BY rings)
FROM delaney_processed

UNION ALL

SELECT 
    'rotatable_bonds',
    AVG(rotatable_bonds), STDDEV(rotatable_bonds), MIN(rotatable_bonds), MAX(rotatable_bonds),
    percentile_cont(0.25) WITHIN GROUP (ORDER BY rotatable_bonds),
    percentile_cont(0.50) WITHIN GROUP (ORDER BY rotatable_bonds),
    percentile_cont(0.75) WITHIN GROUP (ORDER BY rotatable_bonds)
FROM delaney_processed

UNION ALL

SELECT 
    'polar_surface_area',
    AVG(polar_surface_area), STDDEV(polar_surface_area), MIN(polar_surface_area), MAX(polar_surface_area),
    percentile_cont(0.25) WITHIN GROUP (ORDER BY polar_surface_area),
    percentile_cont(0.50) WITHIN GROUP (ORDER BY polar_surface_area),
    percentile_cont(0.75) WITHIN GROUP (ORDER BY polar_surface_area)
FROM delaney_processed

UNION ALL

SELECT 
    'measured_log_solubility',
    AVG(measured_log_solubility), STDDEV(measured_log_solubility), MIN(measured_log_solubility), MAX(measured_log_solubility),
    percentile_cont(0.25) WITHIN GROUP (ORDER BY measured_log_solubility),
    percentile_cont(0.50) WITHIN GROUP (ORDER BY measured_log_solubility),
    percentile_cont(0.75) WITHIN GROUP (ORDER BY measured_log_solubility)
FROM delaney_processed
"""
        result_cursor = await session.execute(text(sql))
        result = result_cursor.fetchall()

    return pd.DataFrame(result)


async def read_delaney_processed():
    db_url = os.environ.get("DB_URL")
    engine = create_async_engine(db_url)

    async with engine.connect() as connection:
        all_df = await fetch_all_delaney_processed(connection)
        stats_df = await fetch_stats_delaney_processed(connection)

    await engine.dispose()
    return all_df, stats_df


all_df, stats_df = await read_delaney_processed()
display(all_df.describe())
display(all_df.pivot(columns=["smiles", "molecular_weight"]))
display(all_df)

display(stats_df)

,esol_predicted_log_solubility,minimum_degree,molecular_weight,hbond_donors,rings,rotatable_bonds,polar_surface_area,measured_log_solubility
count,1128.000000,1128.000000,1128.000000,1128.000000,1128.000000,1128.000000,1128.000000,1128.000000
mean,-2.988192,1.058511,203.937074,0.701241,1.390957,2.177305,34.872881,-3.050102
std,1.683220,0.238560,102.738077,1.089727,1.318286,2.640974,35.383593,2.096441
min,-9.702000,0.000000,16.043000,0.000000,0.000000,0.000000,0.000000,-11.600000
25%,-3.948250,1.000000,121.183000,0.000000,0.000000,0.000000,0.000000,-4.317500
50%,-2.870000,1.000000,182.179000,0.000000,1.000000,1.000000,26.300000,-2.860000
75%,-1.843750,1.000000,270.372000,1.000000,2.000000,3.000000,55.440000,-1.600000
max,1.091000,2.000000,780.949000,11.000000,8.000000,23.000000,268.680000,1.580000


compound_id  \
smiles           OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)C(O)C3O    
molecular_weight                                                457.432   
0                                                         Amigdalin       
1                                                               NaN       
2                                                               NaN       
3                                                               NaN       
4                                                               NaN       
...                                                             ...       
1123                                                            NaN       
1124                                                            NaN       
1125                                                            NaN       
1126                                                            NaN       
1127                                                            NaN       

                                                              \
smiles           Cc1occc1C(=O)Nc2ccccc2 CC(C)=CCCC(C)=CC(=O)   
molecular_weight                201.225              152.237   
0                                   NaN                  NaN   
1                              Fenfuram                  NaN   
2                                   NaN               citral   
3                                   NaN                  NaN   
4                                   NaN                  NaN   
...                                 ...                  ...   
1123                                NaN                  NaN   
1124                                NaN                  NaN   
1125                                NaN                  NaN   
1126                                NaN                  NaN   
1127                                NaN                  NaN   

                                                                               \
smiles           c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43    c1ccsc1 c2ccc1scnc1c2    
molecular_weight                            278.354    84.143         135.191   
0                                               NaN        NaN            NaN   
1                                               NaN        NaN            NaN   
2                                               NaN        NaN            NaN   
3                                            Picene        NaN            NaN   
4                                               NaN  Thiophene            NaN   
...                                             ...        ...            ...   
1123                                            NaN        NaN            NaN   
1124                                            NaN        NaN            NaN   
1125                                            NaN        NaN            NaN   
1126                                            NaN        NaN            NaN   
1127                                            NaN        NaN            NaN   

                                                     \
smiles           Clc1cc(Cl)c(c(Cl)c1)c2c(Cl)cccc2Cl   
molecular_weight                            326.437   
0                                               NaN   
1                                               NaN   
2                                               NaN   
3                                               NaN   
4                                               NaN   
...                                             ...   
1123                                            NaN   
1124                                            NaN   
1125                                            NaN   
1126                                            NaN   
1127                                            NaN   

                                                   \
smiles           CC12CCC3C(CCc4cc(O)ccc34)C2CCC1O   
molecular_weight                          272.388   
0                                             N

,compound_id,esol_predicted_log_solubility,minimum_degree,molecular_weight,hbond_donors,rings,rotatable_bonds,polar_surface_area,measured_log_solubility,smiles
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.770,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.300,Cc1occc1C(=O)Nc2ccccc2
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.060,CC(C)=CCCC(C)=CC(=O)
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.870,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.330,c1ccsc1
...,...,...,...,...,...,...,...,...,...,...
1123,halothane,-2.608,1,197.381,0,0,0,0.00,-1.710,FC(F)(F)C(Cl)Br
1124,Oxamyl,-0.908,1,219.266,1,0,1,71.00,0.106,CNC(=O)ON=C(SC)C(=O)N(C)C
1125,Thiometon,-3.323,1,246.359,0,0,7,18.46,-3.091,CCSCCSP(=S)(OC)OC
1126,2-Methylbutane,-2.245,1,72.151,0,0,1,0.00,-3.180,CCC(C)C


,column_name,mean,std,min,max,p25,p50,p75
0,esol_predicted_log_solubility,-2.988192,1.683220,-9.702,1.091,-3.94825,-2.870,-1.84375
1,minimum_degree,1.058511,0.238560,0.000,2.000,1.00000,1.000,1.00000
2,molecular_weight,203.937074,102.738077,16.043,780.949,121.18300,182.179,270.37200
3,hbond_donors,0.701241,1.089727,0.000,11.000,0.00000,0.000,1.00000
4,rings,1.390957,1.318286,0.000,8.000,0.00000,1.000,2.00000
5,rotatable_bonds,2.177305,2.640974,0.000,23.000,0.00000,1.000,3.00000
6,polar_surface_area,34.872881,35.383593,0.000,268.680,0.00000,26.300,55.44000
7,measured_log_solubility,-3.050102,2.096441,-11.600,1.580,-4.31750,-2.860,-1.60000
